# Φ_p factory — batch modular polynomial production

**Purpose**: compute the classical modular polynomials Φ_p for every prime p ≤ 71 missing from
`pycode/data/classical_modpolys.json` — in particular the non-Atkin primes **37, 43, 53, 61, 67** —
on a machine that can run unattended for a long time.

**Just hit Run All.** Everything checkpoints (the rigid-l-set cache per disc, the CRT state in
`data/phi_factory_state.json` after every prime, each finished Φ_p immediately into
`classical_modpolys.json`), so you can interrupt and re-run at any point; finished targets are skipped.

**What it does per prime ℓ** (ascending, starting from the shipped ℓ < 1024 data): extend the
rigid-l-set cache to cover the class discs at ℓ; compute the j ↔ form bijections of every class
(skipping rigid-gap discs); extract p-isogenous pairs for each unfinished target; solve the linear
system for Φ_p mod ℓ (diagonal = −∏H_d from the Hilbert library + pair evaluations); CRT-accumulate.
A target finishes when its modulus exceeds the provable Bröker–Sutherland height bound, passes the
Kronecker congruence check, and is saved.

**Expected behaviour**: small targets (29, 31) finish first, from primes ℓ ≳ 750;
p = 67, 71 only start collecting at ℓ ≈ 4000–4500 (a prime needs ≈ (p+1)(p+2)/2 equations and
supplies ≈ 0.55·ℓ pairs) and finish around ℓ ≈ 11000–14000. Expect roughly a minute per fresh prime
(bijections) plus solver time that grows with p — **a run of a day or two in total**. Occasional
`[rigid cache: extending...]` blocks take a few minutes each.

**Getting results back**: the run writes `data/classical_modpolys.json` (the polynomials),
`data/rigid_lset_cache.json` (reusable per-disc data) and `data/phi_factory_state.json`
(resume state) — commit those and pull them into the main machine.

In [ ]:
import os
os.chdir('../pycode')
from phi_factory import *

# optional GPU trace tables (needs pytorch; falls back to numpy silently)
try:
    import trace_gpu
    print('trace backend:', trace_gpu.enable())
except ImportError:
    print('trace backend: numpy (pip install torch to enable cuda/mps)')

targets = factory_targets()
print('missing modular polynomials for p =', targets)
print('priority (non-Atkin):', [p for p in targets if p in WANTED])

In [ ]:
# preflight: the diagonal support Hilbert polynomials must all be known
from modpoly_crt import phi_diagonal
for p in targets:
    phi_diagonal(p)          # raises with the missing discs if the library falls short
print('diagonal support: complete for all targets')

In [ ]:
# the long run -- safe to interrupt and re-run at any time
run_factory(targets)

In [ ]:
# quick sanity view of whatever has finished so far
from modularpolynomials import _modpoly_cache
from modpoly_crt import kronecker_check
done = sorted(p for p in targets if p in _modpoly_cache)
for p in done:
    print(f'Phi_{p}: kronecker mod p:', kronecker_check(p, _modpoly_cache[p]))
print('finished:', done)